# Activity: Foundational Statistical Outlier Detection
# **SOLUTION VERSION**

## Learning Objectives

By the end of this activity, you will be able to:

1. Identify outliers using visualization techniques (histograms, box plots, violin plots)
2. Implement and apply Tukey's IQR method for outlier detection
3. Detect outliers using Z-score and Modified Z-score methods
4. Understand the limitations of univariate outlier detection
5. Apply multivariate outlier detection using Mahalanobis Distance and Isolation Forest
6. Compare and contrast different outlier detection methods

## Dataset

We'll use a weight-height dataset containing measurements from individuals. This dataset is ideal for learning outlier detection because:
- It has clear univariate outliers (extreme values in single variables)
- It demonstrates multivariate outliers (unusual combinations of values)
- The data is intuitive and easy to interpret

## Setup and Imports

In [ ]:
# Check library versions
import matplotlib 
import pandas as pd
import scipy 
import statsmodels

print(f'''
matplotlib -> {matplotlib.__version__}
pandas -> {pd.__version__}   
scipy -> {scipy.__version__}
statsmodels -> {statsmodels.__version__}
''')

In [ ]:
# Main imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import scipy.stats as stats
from sklearn.ensemble import IsolationForest
from scipy.spatial.distance import mahalanobis

# Set visualization defaults
plt.rcParams["figure.figsize"] = [12, 5]
sns.set_style("whitegrid")

## Load and Explore Data

In [ ]:
# Load the weight-height dataset
file = Path("../data/weight-height.csv")
wh = pd.read_csv(file)

# Display basic information
print(f"Dataset shape: {wh.shape}")
print(f"\nFirst few rows:")
wh.head()

In [ ]:
# Summary statistics
wh.describe()

In [ ]:
# Check distribution of Gender
wh['Gender'].value_counts()

---

# Recipe 1: Detecting Outliers using Visualization

## Learning Objectives
- Understand how different visualizations reveal outliers
- Learn when to use histograms vs. box plots vs. violin plots
- Recognize the subjective nature of visual outlier detection

## Key Concepts

Visual inspection is often the first step in outlier detection. Different plots reveal different aspects:
- **Histograms**: Show distribution shape and extreme values
- **Box plots**: Highlight statistical outliers using quartiles
- **Violin plots**: Combine distribution shape with quartile information

## 1.1 Histogram Visualization

In [ ]:
# Simple histogram of both variables
sns.histplot(wh)
plt.title('Distribution of Height and Weight')
plt.show()

In [ ]:
# Separate histograms with better layout
g = sns.displot(wh, kind='hist', height=5, aspect=2)
g.fig.suptitle('Distribution of Height and Weight', y=1.02)
plt.show()

## 1.2 Box Plot Visualization

Box plots show:
- **Box**: Interquartile range (IQR, 25th to 75th percentile)
- **Line in box**: Median
- **Whiskers**: Typically extend to 1.5 × IQR
- **Points beyond whiskers**: Outliers

In [ ]:
# Box plot for both variables
sns.boxplot(wh, orient='h', whis=1.5)
plt.title('Box Plot of Height and Weight (1.5 × IQR)')
plt.show()

In [ ]:
# Individual box plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(wh['Height'], orient='h', whis=1.5, ax=axes[0])
axes[0].set_title('Height Distribution')

sns.boxplot(wh['Weight'], orient='h', whis=1.5, ax=axes[1])
axes[1].set_title('Weight Distribution')

plt.tight_layout()
plt.show()

## 1.3 Violin Plot Visualization

Violin plots combine the box plot with a kernel density estimation.

In [ ]:
# Violin plot showing quartiles
sns.violinplot(wh, inner='quartile', orient='h')
plt.title('Violin Plot of Height and Weight')
plt.show()

### Exercise 1.1: Visual Analysis - **SOLUTION**

Based on the visualizations above, answer these questions:

1. Which variable (Height or Weight) appears to have more outliers?
2. Do the outliers appear symmetric (both high and low) or skewed to one side?
3. What are the advantages and disadvantages of visual outlier detection?

**SOLUTION:**
- **Question 1:** Both Height and Weight appear to have similar numbers of outliers when using the 1.5×IQR rule shown in the box plots. Weight might have slightly more extreme outliers on both ends.

- **Question 2:** The outliers appear fairly symmetric for both variables - there are extreme values on both the low and high ends. This is expected for biological measurements like height and weight in a population.

- **Question 3:** 
  - **Advantages:** Visual methods are intuitive, easy to interpret, don't require threshold decisions upfront, and help understand data distribution. They're great for exploratory analysis and communication.
  - **Disadvantages:** Subjective (different people might identify different outliers), doesn't scale well to high dimensions, doesn't provide quantitative measures, and can't be easily automated. Visual inspection alone isn't sufficient for formal analysis.

---

# Recipe 2: Detecting Outliers using Tukey's Method (IQR)

## Learning Objectives
- Understand Tukey's fence method for outlier detection
- Implement the IQR calculation and boundary determination
- Learn how to adjust sensitivity using the k parameter

## Key Concepts

**Tukey's Method** defines outliers as points that fall outside these fences:
- **Lower fence**: Q1 - k × IQR
- **Upper fence**: Q3 + k × IQR

Where:
- Q1 = 25th percentile, Q3 = 75th percentile
- IQR = Q3 - Q1 (Interquartile Range)
- k = sensitivity parameter (commonly 1.5 for outliers, 3.0 for extreme outliers)

## Understanding Percentiles and Quantiles

In [ ]:
# Examine key percentiles
percentiles = [0, 5, 10, 25, 50, 75, 90, 95, 100]
weight_percentiles = np.percentile(wh['Weight'], percentiles)

print("Weight Percentiles:")
for p, v in zip(percentiles, weight_percentiles):
    print(f"  {p:3d}th percentile: {v:.2f}")

## Implementation: Tukey's Method - **SOLUTION**

Complete implementation of the `iqr_outliers` function.

In [ ]:
def iqr_outliers(data, k=1.5):
    """
    Detect outliers using Tukey's method with customizable fence multiplier.
    
    Parameters:
    -----------
    data : pandas.Series or numpy.array
        The data to analyze for outliers
    k : float, default=1.5
        The fence multiplier (1.5 for outliers, 3.0 for extreme outliers)
    
    Returns:
    --------
    pandas.Series or numpy.array
        Data points identified as outliers
    """
    # SOLUTION: Calculate Q1 and Q3 using np.percentile
    q1, q3 = np.percentile(data, [25, 75])
    
    # SOLUTION: Calculate the IQR
    IQR = q3 - q1
    
    # SOLUTION: Calculate lower and upper fences
    lower_fence = q1 - k * IQR
    upper_fence = q3 + k * IQR
    
    # SOLUTION: Return data points outside the fences
    return data[(data < lower_fence) | (data > upper_fence)]

## Test Your Implementation

In [ ]:
# Test on Height with k=1.5
print("Testing IQR method on Height:")
sns.boxplot(wh['Height'], orient='h', whis=1.5)
plt.title('Height Box Plot (k=1.5)')
plt.show()

height_outliers = iqr_outliers(wh['Height'], k=1.5)
print(f"\nNumber of Height outliers detected: {len(height_outliers)}")
print(f"Outlier values: {height_outliers.values[:10]}...")  # Show first 10

In [ ]:
# Test on Weight with k=1.5
print("Testing IQR method on Weight:")
sns.boxplot(wh['Weight'], orient='h', whis=1.5)
plt.title('Weight Box Plot (k=1.5)')
plt.show()

weight_outliers = iqr_outliers(wh['Weight'], k=1.5)
print(f"\nNumber of Weight outliers detected: {len(weight_outliers)}")
print(f"Outlier values: {weight_outliers.values[:10]}...")  # Show first 10

### Exercise 2.1: Sensitivity Analysis - **SOLUTION**

Test how the number of detected outliers changes with different k values:

In [ ]:
# SOLUTION: Test with k=1.0, 1.5, 2.0, 3.0
k_values = [1.0, 1.5, 2.0, 3.0]

print("Sensitivity Analysis - Height:")
for k in k_values:
    outliers = iqr_outliers(wh['Height'], k=k)
    print(f"k={k}: {len(outliers)} outliers detected")

**Question:** What happens to the number of outliers as k increases? Why?

**SOLUTION:** As k increases, the number of detected outliers decreases. This is because:
- Larger k values create wider fences (further from Q1 and Q3)
- Wider fences mean only more extreme values are classified as outliers
- k=1.5 is the standard choice and detects "mild outliers"
- k=3.0 detects only "extreme outliers"
- The choice of k should depend on your application's tolerance for outliers and the consequences of false positives vs. false negatives

---

# Recipe 3: Detecting Outliers using Z-Scores

## Learning Objectives
- Understand the Z-score method and its assumptions
- Implement Z-score calculation for outlier detection
- Learn when Z-score method is appropriate (normal distributions)

## Key Concepts

**Z-score** measures how many standard deviations a point is from the mean:

$$z = \frac{x - \mu}{\sigma}$$

Where:
- x = data point
- μ = mean
- σ = standard deviation

**Common thresholds:**
- |z| > 2: ~5% of data (95% within)
- |z| > 3: ~0.3% of data (99.7% within)

**Assumption:** Data should be approximately normally distributed

## Implementation: Z-Score Detection - **SOLUTION**

In [ ]:
def zscore(df, threshold=3):
    """
    Detect outliers using z-score method with customizable threshold.
    
    Parameters:
    -----------
    df : pandas.Series
        Data to analyze for outliers
    threshold : float, default=3
        The threshold in standard deviations (typically 2-3)
    
    Returns:
    --------
    tuple: (outliers, transformed)
        - outliers: DataFrame containing outlier points with their z-scores
        - transformed: Full DataFrame with z-scores column added
    """
    data = df.copy()
    
    # SOLUTION: Calculate z-score for each data point
    data['zscore'] = (data - data.mean()) / data.std()
    
    # SOLUTION: Identify outliers where |z-score| > threshold
    outliers = data[(data['zscore'] < -threshold) | (data['zscore'] > threshold)]
    
    return outliers, data

## Helper Function for Visualization

This function is provided to help you visualize z-scores:

In [ ]:
def plot_zscore(data_series, d=3, title='Standardized Data with Outlier Thresholds'):
    """
    Plot the standardized z-scores with threshold lines.
    
    Parameters:
    -----------
    data_series : pandas.Series
        Series containing z-scores
    d : float, default=3
        Threshold in standard deviations
    title : str
        Plot title
    """
    plt.figure(figsize=(12, 5))
    plt.plot(data_series.index, data_series.values, 'k^', markersize=4, alpha=0.6)
    
    plt.axhline(y=d, color='r', linestyle='--', label=f'+{d} SD', linewidth=2)
    plt.axhline(y=-d, color='r', linestyle='--', label=f'-{d} SD', linewidth=2)
    plt.axhline(y=0, color='gray', linestyle='-', alpha=0.3, linewidth=1)
    
    # Highlight outliers
    outliers = data_series[abs(data_series) > d]
    if not outliers.empty:
        plt.plot(outliers.index, outliers.values, 'ro', markersize=8, label='Outliers', alpha=0.7)
    
    plt.ylabel('Z-score')
    plt.xlabel('Data Point Index')
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

## Test Your Implementation

In [ ]:
# Apply z-score method to Height with threshold=3
outliers, transformed = zscore(wh['Height'], threshold=3)

print(f"Number of outliers detected: {len(outliers)}")
print(f"\nOutlier statistics:")
print(outliers.describe())

In [ ]:
# Visualize the z-scores
plot_zscore(transformed['zscore'], d=3.0, title='Height Z-Scores (threshold=3)')

In [ ]:
# Check distribution of z-scores
transformed['zscore'].hist(bins=50, edgecolor='black')
plt.axvline(x=3, color='r', linestyle='--', label='Threshold=3')
plt.axvline(x=-3, color='r', linestyle='--')
plt.xlabel('Z-score')
plt.ylabel('Frequency')
plt.title('Distribution of Z-Scores')
plt.legend()
plt.show()

### Exercise 3.1: Compare Thresholds - **SOLUTION**

In [ ]:
# SOLUTION: Compare outlier detection with thresholds 2, 2.5, and 3
thresholds = [2, 2.5, 3]

print("Threshold Comparison - Height:")
for t in thresholds:
    outliers, _ = zscore(wh['Height'], threshold=t)
    print(f"Threshold={t}: {len(outliers)} outliers detected")

### Exercise 3.2: Test Normality Assumption - **SOLUTION**

The z-score method assumes normal distribution. Let's test this:

In [ ]:
from statsmodels.stats.diagnostic import kstest_normal

# Kolmogorov-Smirnov test for normality
def test_normal(data):
    result = kstest_normal(data)
    print(f"KS Statistic: {result[0]:.4f}")
    print(f"P-value: {result[1]:.4f}")
    if result[1] > 0.05:
        print("Conclusion: Data appears normally distributed (p > 0.05)")
    else:
        print("Conclusion: Data does NOT appear normally distributed (p ≤ 0.05)")

print("Normality Test for Height:")
test_normal(wh['Height'])

**Question:** Is the Height data normally distributed? What does this mean for using Z-scores?

**SOLUTION:** Based on the KS test, if p > 0.05, the data appears to be normally distributed, which means the z-score method is appropriate. Height data from a large population sample typically follows a normal distribution reasonably well. 

However, if the data were NOT normally distributed (p ≤ 0.05), the z-score assumptions would be violated, and we should:
- Consider using Modified Z-score instead (which is more robust)
- Use non-parametric methods like IQR
- Transform the data to achieve normality (if appropriate)
- Be cautious about interpreting z-score thresholds literally

---

# Recipe 4: Detecting Outliers using Modified Z-Score

## Learning Objectives
- Understand the limitations of standard z-score for non-normal data
- Implement the Modified Z-score using median and MAD
- Compare robustness of Modified Z-score vs. standard Z-score

## Key Concepts

**Modified Z-score** is more robust to outliers because it uses:
- **Median** instead of mean (not affected by extreme values)
- **MAD (Median Absolute Deviation)** instead of standard deviation

$$M_i = \frac{0.6745(x_i - \text{median})}{\text{MAD}}$$

Where:
- MAD = median(|x - median(x)|)
- 0.6745 ≈ Φ⁻¹(0.75) makes MAD comparable to standard deviation

**Threshold:** Typically |M| > 3.5 or |M| > 2.5

## Understanding the Scaling Factor

The 0.6745 factor comes from the standard normal distribution's 75th percentile:

In [ ]:
# The scaling factor links MAD to standard deviation
scaling_factor = stats.norm.ppf(0.75)
print(f"Scaling factor (75th percentile of standard normal): {scaling_factor:.4f}")
print(f"\nThis makes MAD comparable to standard deviation for normal distributions")

## Implementation: Modified Z-Score - **SOLUTION**

In [ ]:
def modified_zscore(df, threshold=3.5):
    """
    Detect outliers using modified z-score method with customizable threshold.
    
    Parameters:
    -----------
    df : pandas.Series
        Data to analyze for outliers
    threshold : float, default=3.5
        The threshold for modified z-scores (typically 2.5-3.5)
    
    Returns:
    --------
    tuple: (outliers, transformed)
        - outliers: DataFrame containing outlier points with modified z-scores
        - transformed: Full DataFrame with modified z-scores column added
    """
    data = df.copy()
    
    # SOLUTION: Calculate median of the data
    median = np.median(data)
    
    # SOLUTION: Calculate MAD (Median Absolute Deviation)
    MAD = np.median(np.abs(data - median))
    
    # SOLUTION: Calculate scaling factor
    s = stats.norm.ppf(0.75)
    
    # SOLUTION: Calculate modified z-score
    data['m_zscore'] = s * (data - median) / MAD
    
    # SOLUTION: Identify outliers where |modified z-score| > threshold
    outliers = data[(data['m_zscore'] < -threshold) | (data['m_zscore'] > threshold)]
    
    return outliers, data

## Test Your Implementation

In [ ]:
# Apply modified z-score method with threshold=2.5
outliers_mod, transformed_mod = modified_zscore(wh['Height'], threshold=2.5)

print(f"Number of outliers detected: {len(outliers_mod)}")
print(f"\nOutlier statistics:")
print(outliers_mod.describe())

In [ ]:
# Visualize modified z-scores
plot_zscore(transformed_mod['m_zscore'], d=2.5, title='Height Modified Z-Scores (threshold=2.5)')

In [ ]:
# Compare distribution of modified z-scores
transformed_mod['m_zscore'].hist(bins=50, edgecolor='black')
plt.axvline(x=2.5, color='r', linestyle='--', label='Threshold=2.5')
plt.axvline(x=-2.5, color='r', linestyle='--')
plt.xlabel('Modified Z-score')
plt.ylabel('Frequency')
plt.title('Distribution of Modified Z-Scores')
plt.legend()
plt.show()

### Exercise 4.1: Compare Standard vs Modified Z-Score - **SOLUTION**

In [ ]:
# Compare standard z-score (threshold=3) vs modified z-score (threshold=2.5)
outliers_standard, _ = zscore(wh['Height'], threshold=3)
outliers_modified, _ = modified_zscore(wh['Height'], threshold=2.5)

print("Comparison of Methods:")
print(f"Standard Z-score (t=3):  {len(outliers_standard)} outliers")
print(f"Modified Z-score (t=2.5): {len(outliers_modified)} outliers")

# Check overlap
overlap = set(outliers_standard.index) & set(outliers_modified.index)
print(f"\nOverlapping outliers: {len(overlap)}")

**Question:** Why might the Modified Z-score detect different outliers than standard Z-score?

**SOLUTION:** The Modified Z-score can detect different outliers because:

1. **Robustness to outliers:** Modified Z-score uses median and MAD, which are not influenced by extreme values. Standard Z-score uses mean and standard deviation, which ARE influenced by outliers. This means outliers themselves can affect the calculation that's supposed to detect them!

2. **Different reference points:** The median (used by Modified Z-score) and mean (used by standard Z-score) can be different, especially in skewed distributions. This shifts what's considered "center."

3. **Different spread measures:** MAD vs. standard deviation measure spread differently. Standard deviation is more affected by extreme values, potentially making them appear less extreme.

4. **Threshold differences:** We typically use different thresholds (3 for standard, 2.5-3.5 for modified), which reflects their different scales.

**Instructor Note:** Modified Z-score is generally preferred when you suspect outliers might contaminate your mean and standard deviation calculations.

---

# Recipe 5: Multivariate Outlier Detection (NEW)

## Learning Objectives
- Understand why univariate methods fail on multivariate outliers
- Implement Mahalanobis Distance for multivariate outlier detection
- Apply Isolation Forest for anomaly detection
- Compare multivariate vs univariate approaches

## Key Concepts

**The Problem:** A data point can be an outlier in the multivariate space even if each individual feature is normal!

Example: A person who is 5'2" and weighs 200 lbs:
- Height alone: Not necessarily an outlier
- Weight alone: Not necessarily an outlier
- **Combination**: Definitely unusual!

### Two Approaches:

1. **Mahalanobis Distance**: Measures distance from center accounting for correlations
2. **Isolation Forest**: Uses decision trees to isolate anomalies

## 5.1 Visualize the Multivariate Distribution

In [ ]:
# Create a scatter plot to see the relationship between Height and Weight
plt.figure(figsize=(10, 6))
plt.scatter(wh['Height'], wh['Weight'], alpha=0.5, s=10)
plt.xlabel('Height (inches)')
plt.ylabel('Weight (pounds)')
plt.title('Height vs Weight Distribution')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Correlation between Height and Weight: {wh['Height'].corr(wh['Weight']):.3f}")

## 5.2 Understanding Mahalanobis Distance

**Mahalanobis Distance** accounts for correlations between variables:

$$D_M(x) = \sqrt{(x - \mu)^T \Sigma^{-1} (x - \mu)}$$

Where:
- x = data point
- μ = mean vector
- Σ = covariance matrix

**Intuition:** It measures how many "standard deviations" away a point is, considering correlations.

## Implementation: Mahalanobis Distance - **SOLUTION**

In [ ]:
def detect_mahalanobis_outliers(df, threshold=3):
    """
    Detect outliers using Mahalanobis Distance.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Data with multiple numeric columns
    threshold : float, default=3
        Threshold for Mahalanobis distance (chi-square based)
    
    Returns:
    --------
    tuple: (outliers, distances)
        - outliers: DataFrame of outlier points
        - distances: Series of Mahalanobis distances for all points
    """
    # SOLUTION: Calculate mean vector of the data
    mean = df.mean()
    
    # SOLUTION: Calculate covariance matrix
    cov_matrix = df.cov()
    
    # SOLUTION: Calculate inverse of covariance matrix
    inv_cov = np.linalg.inv(cov_matrix)
    
    # Calculate Mahalanobis distance for each point
    distances = []
    for idx in df.index:
        # SOLUTION: Get the data point as array
        point = df.loc[idx].values
        
        # SOLUTION: Calculate Mahalanobis distance
        dist = mahalanobis(point, mean, inv_cov)
        distances.append(dist)
    
    # Create series of distances
    distances = pd.Series(distances, index=df.index, name='mahalanobis_distance')
    
    # Determine threshold based on chi-square distribution
    # For 2 degrees of freedom (2 variables), threshold^2 follows chi-square
    chi2_threshold = stats.chi2.ppf(0.95, df=df.shape[1])
    distance_threshold = np.sqrt(chi2_threshold)
    
    print(f"Using distance threshold: {distance_threshold:.2f} (95th percentile)")
    
    # SOLUTION: Identify outliers where distance > threshold
    outlier_mask = distances > distance_threshold
    outliers = df[outlier_mask].copy()
    outliers['mahalanobis_distance'] = distances[outlier_mask]
    
    return outliers, distances

## Test Mahalanobis Distance

In [ ]:
# Apply Mahalanobis distance to Height and Weight
outliers_mahal, distances_mahal = detect_mahalanobis_outliers(wh[['Height', 'Weight']])

print(f"Number of outliers detected: {len(outliers_mahal)}")
print(f"\nTop 5 outliers by Mahalanobis distance:")
print(outliers_mahal.nlargest(5, 'mahalanobis_distance'))

In [ ]:
# Visualize Mahalanobis distances
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: Scatter plot with outliers highlighted
axes[0].scatter(wh['Height'], wh['Weight'], c='blue', alpha=0.3, s=10, label='Normal')
axes[0].scatter(outliers_mahal['Height'], outliers_mahal['Weight'], 
                c='red', s=50, label='Outliers', edgecolors='black', linewidth=1)
axes[0].set_xlabel('Height (inches)')
axes[0].set_ylabel('Weight (pounds)')
axes[0].set_title('Mahalanobis Outliers in 2D Space')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: Distribution of distances
axes[1].hist(distances_mahal, bins=50, edgecolor='black')
chi2_threshold = np.sqrt(stats.chi2.ppf(0.95, df=2))
axes[1].axvline(x=chi2_threshold, color='r', linestyle='--', 
                label=f'Threshold={chi2_threshold:.2f}', linewidth=2)
axes[1].set_xlabel('Mahalanobis Distance')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Mahalanobis Distances')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5.3 Isolation Forest

**Isolation Forest** works differently:
- Builds random decision trees
- Anomalies are easier to "isolate" (require fewer splits)
- Returns an anomaly score: -1 for outliers, 1 for inliers

**Advantages:**
- No assumptions about data distribution
- Scales well to high dimensions
- Can handle complex, non-linear relationships

## Implementation: Isolation Forest Detection - **SOLUTION**

In [ ]:
def detect_isolation_forest_outliers(df, contamination=0.05, random_state=42):
    """
    Detect outliers using Isolation Forest.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Data with numeric columns
    contamination : float, default=0.05
        Expected proportion of outliers (0.01 to 0.5)
    random_state : int, default=42
        Random seed for reproducibility
    
    Returns:
    --------
    tuple: (outliers, predictions, scores)
        - outliers: DataFrame of detected outliers
        - predictions: Array of predictions (-1 for outliers, 1 for inliers)
        - scores: Array of anomaly scores (lower = more anomalous)
    """
    # SOLUTION: Create an Isolation Forest model
    iso_forest = IsolationForest(contamination=contamination, random_state=random_state)
    
    # SOLUTION: Fit the model and predict
    predictions = iso_forest.fit_predict(df)
    
    # SOLUTION: Get anomaly scores
    scores = iso_forest.score_samples(df)
    
    # SOLUTION: Create DataFrame of outliers (where prediction == -1)
    outlier_mask = predictions == -1
    outliers = df[outlier_mask].copy()
    outliers['anomaly_score'] = scores[outlier_mask]
    
    return outliers, predictions, scores

## Test Isolation Forest

In [ ]:
# Apply Isolation Forest
outliers_iso, predictions_iso, scores_iso = detect_isolation_forest_outliers(
    wh[['Height', 'Weight']], 
    contamination=0.05
)

print(f"Number of outliers detected: {len(outliers_iso)}")
print(f"Percentage of data: {len(outliers_iso)/len(wh)*100:.2f}%")
print(f"\nTop 5 outliers by anomaly score (most anomalous):")
print(outliers_iso.nsmallest(5, 'anomaly_score'))

In [ ]:
# Visualize Isolation Forest results
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: Scatter plot with outliers highlighted
normal_data = wh[predictions_iso == 1]
axes[0].scatter(normal_data['Height'], normal_data['Weight'], 
                c='blue', alpha=0.3, s=10, label='Normal')
axes[0].scatter(outliers_iso['Height'], outliers_iso['Weight'], 
                c='red', s=50, label='Outliers', edgecolors='black', linewidth=1)
axes[0].set_xlabel('Height (inches)')
axes[0].set_ylabel('Weight (pounds)')
axes[0].set_title('Isolation Forest Outliers in 2D Space')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: Distribution of anomaly scores
axes[1].hist(scores_iso, bins=50, edgecolor='black')
axes[1].axvline(x=outliers_iso['anomaly_score'].max(), color='r', 
                linestyle='--', label='Threshold', linewidth=2)
axes[1].set_xlabel('Anomaly Score (lower = more anomalous)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Anomaly Scores')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5.4 Compare All Multivariate Methods

In [ ]:
# Compare Mahalanobis vs Isolation Forest
print("Method Comparison:")
print(f"Mahalanobis Distance: {len(outliers_mahal)} outliers")
print(f"Isolation Forest:     {len(outliers_iso)} outliers")

# Check overlap
overlap = set(outliers_mahal.index) & set(outliers_iso.index)
print(f"\nOverlapping outliers: {len(overlap)}")
print(f"Agreement: {len(overlap)/max(len(outliers_mahal), len(outliers_iso))*100:.1f}%")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# All data
axes[0].scatter(wh['Height'], wh['Weight'], c='lightgray', alpha=0.5, s=10)
axes[0].set_xlabel('Height (inches)')
axes[0].set_ylabel('Weight (pounds)')
axes[0].set_title('All Data')
axes[0].grid(True, alpha=0.3)

# Mahalanobis outliers
axes[1].scatter(wh['Height'], wh['Weight'], c='lightgray', alpha=0.3, s=10)
axes[1].scatter(outliers_mahal['Height'], outliers_mahal['Weight'], 
                c='red', s=50, label='Mahalanobis', edgecolors='black', linewidth=1)
axes[1].set_xlabel('Height (inches)')
axes[1].set_ylabel('Weight (pounds)')
axes[1].set_title(f'Mahalanobis Distance ({len(outliers_mahal)} outliers)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Isolation Forest outliers
axes[2].scatter(wh['Height'], wh['Weight'], c='lightgray', alpha=0.3, s=10)
axes[2].scatter(outliers_iso['Height'], outliers_iso['Weight'], 
                c='blue', s=50, label='Isolation Forest', edgecolors='black', linewidth=1)
axes[2].set_xlabel('Height (inches)')
axes[2].set_ylabel('Weight (pounds)')
axes[2].set_title(f'Isolation Forest ({len(outliers_iso)} outliers)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Exercise 5.1: Why Multivariate Matters - **SOLUTION**

Let's demonstrate why univariate methods miss multivariate outliers:

In [ ]:
# Get univariate outliers
height_outliers_z, _ = zscore(wh['Height'], threshold=3)
weight_outliers_z, _ = zscore(wh['Weight'], threshold=3)
univariate_outliers = set(height_outliers_z.index) | set(weight_outliers_z.index)

# Get multivariate outliers
multivariate_outliers = set(outliers_mahal.index)

# Find outliers ONLY detected by multivariate method
only_multivariate = multivariate_outliers - univariate_outliers

print(f"Univariate outliers (Z-score):    {len(univariate_outliers)}")
print(f"Multivariate outliers (Mahalanobis): {len(multivariate_outliers)}")
print(f"\nOutliers ONLY found by multivariate method: {len(only_multivariate)}")
print(f"\nExample points missed by univariate methods:")
print(wh.loc[list(only_multivariate)][:5])

**Question:** Why do these points appear normal in univariate analysis but unusual in multivariate analysis?

**SOLUTION:** These points appear normal individually but unusual together because:

1. **Correlation matters:** Height and weight are positively correlated in the population. Someone who is 5'2" and weighs 180 lbs might not have an outlier height (5'2" is short but not extremely so) or outlier weight (180 lbs is heavy but not extreme), but the *combination* is unusual given the expected relationship.

2. **Context of the relationship:** Univariate methods ignore relationships between variables. They ask "Is this height unusual?" and "Is this weight unusual?" separately. Multivariate methods ask "Is this height-weight combination unusual?"

3. **The covariance structure:** Mahalanobis distance accounts for the covariance between variables. A point might fall within normal ranges for both variables but be far from the typical pattern when you consider how the variables relate to each other.

**Real-world example:** In fraud detection, a single transaction amount might not be unusual, and the transaction time might not be unusual, but a large transaction at 3 AM is suspicious. Multivariate methods catch these contextual anomalies.

**Instructor Note:** This is why multivariate methods are essential for real-world anomaly detection where features are correlated.

### Exercise 5.2: Contamination Sensitivity - **SOLUTION**

Test how the contamination parameter affects Isolation Forest:

In [ ]:
# SOLUTION: Test contamination values: 0.01, 0.05, 0.10
contamination_values = [0.01, 0.05, 0.10]

print("Contamination Sensitivity Analysis:")
for c in contamination_values:
    outliers, _, _ = detect_isolation_forest_outliers(wh[['Height', 'Weight']], contamination=c)
    print(f"Contamination={c}: {len(outliers)} outliers ({len(outliers)/len(wh)*100:.2f}%)")

---

# Reflection Questions - **SOLUTION**

Answer these questions based on your experience with the activity:

## 1. Method Selection
**Question:** When would you choose each method?

**SOLUTION:**
- **Tukey's IQR:** When you have univariate data that may not be normally distributed, or when you want a distribution-free method. Great for exploratory analysis and when interpretability is important. Works well with skewed data.

- **Z-score:** When your data is approximately normally distributed and you understand the 68-95-99.7 rule. Good when you can verify normality assumption. Simple and widely understood.

- **Modified Z-score:** When you suspect outliers might contaminate your mean/std calculations, or when working with non-normal data. More robust than standard Z-score. Good compromise between robustness and statistical properties.

- **Mahalanobis Distance:** When you have multivariate data with correlated features and approximately normal distributions. Essential when relationships between variables matter. Good for detecting contextual anomalies.

- **Isolation Forest:** When you have high-dimensional data, complex non-linear relationships, or no distributional assumptions. Scales well and handles mixed types of anomalies. Good when you don't know what "normal" looks like mathematically.

## 2. Assumptions and Limitations
**Question:** What assumptions does each method make?

**SOLUTION:**
- **IQR:** 
  - Assumes: Univariate data, symmetry is helpful but not required
  - Limitations: Only considers one variable at a time, sensitive to k parameter choice, may flag too many points in long-tailed distributions

- **Z-score:** 
  - Assumes: Normal distribution, independence of observations
  - Limitations: Mean and std are sensitive to outliers (circular problem), poor performance on non-normal data, only univariate

- **Modified Z-score:** 
  - Assumes: Univariate data, median-based measures are appropriate
  - Limitations: Still univariate, requires sufficient sample size for stable median/MAD estimates

- **Mahalanobis:** 
  - Assumes: Multivariate normal distribution, linear relationships, invertible covariance matrix
  - Limitations: Sensitive to high dimensions (curse of dimensionality), requires sufficient samples to estimate covariance reliably, assumes elliptical contours

- **Isolation Forest:** 
  - Assumes: Outliers are "few and different" (isolation principle)
  - Limitations: Requires setting contamination parameter, randomness means results can vary, less interpretable than statistical methods, may struggle with local outliers

## 3. Practical Considerations
**Question:** What factors should you consider when choosing a threshold?

**SOLUTION:**
- **Domain context:** What are the consequences of false positives vs. false negatives? In medical diagnosis, false negatives are costly. In manufacturing quality control, false positives waste resources.

- **Data size:** With larger samples, you expect more extreme values just by chance. You might need stricter thresholds.

- **Downstream actions:** Will you automatically remove outliers, investigate manually, or just flag for review? This affects acceptable false positive rates.

- **Expected outlier rate:** If you know ~1% of data should be outliers (e.g., fraud rate), calibrate accordingly.

- **Computational resources:** Some methods (especially multivariate) are more computationally expensive.

- **Interpretability needs:** Statistical methods provide clear explanations ("3 standard deviations from mean"), ML methods are more black-box.

- **Historical baselines:** What thresholds have worked well in similar applications?

## 4. Multivariate vs Univariate
**Question:** Why is it important to use multivariate methods when analyzing datasets with multiple features?

**SOLUTION:**
Multivariate methods are crucial because:

1. **Correlation structure:** Real-world features are rarely independent. Multivariate methods exploit the relationships between features to detect contextual anomalies that univariate methods miss.

2. **Contextual anomalies:** A data point can be normal on every individual feature but abnormal in combination. Example: Your credit card spending $500 is normal, shopping at 2 AM is normal, but $500 at 2 AM might be fraud.

3. **Efficiency:** Running 10 univariate tests has higher false positive rates than one multivariate test (multiple testing problem).

4. **Complete picture:** Features often have meaning only in context. Height alone doesn't tell you much about health without weight, age, etc.

5. **Real-world complexity:** Most interesting phenomena are multivariate. Customer behavior, system health, disease diagnosis, etc., all depend on multiple interacting factors.

**However:** Univariate methods are still valuable for initial exploration, interpretability, and when features are truly independent.

**Best practice:** Start with univariate analysis to understand each feature, then move to multivariate methods for comprehensive detection.

---

# Challenge Exercise: Apply to New Dataset - **SOLUTION**

Now apply what you've learned to a synthetic dataset with known outliers!

## Generate Challenge Dataset

In [ ]:
# Generate synthetic data with known outliers
np.random.seed(42)

# Normal data
n_samples = 1000
mean = [50, 100]
cov = [[10, 8], [8, 20]]  # Correlated variables
normal_data = np.random.multivariate_normal(mean, cov, n_samples)

# Add outliers
n_outliers = 50
outlier_data = np.random.uniform([30, 50], [70, 150], (n_outliers, 2))

# Combine
all_data = np.vstack([normal_data, outlier_data])
labels = np.array([0] * n_samples + [1] * n_outliers)  # 0=normal, 1=outlier

# Create DataFrame
challenge_df = pd.DataFrame(all_data, columns=['Feature_1', 'Feature_2'])
challenge_df['true_label'] = labels

print(f"Challenge dataset shape: {challenge_df.shape}")
print(f"True outliers: {n_outliers} ({n_outliers/len(challenge_df)*100:.1f}%)")

In [ ]:
# Visualize the challenge dataset
plt.figure(figsize=(10, 6))
colors = ['blue' if label == 0 else 'red' for label in challenge_df['true_label']]
plt.scatter(challenge_df['Feature_1'], challenge_df['Feature_2'], 
            c=colors, alpha=0.5, s=20)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Challenge Dataset (Red = True Outliers)')
plt.grid(True, alpha=0.3)
plt.show()

## Challenge Tasks - **SOLUTION**

We'll apply multiple methods and compare their performance.

In [ ]:
# Helper function to calculate metrics
def calculate_metrics(true_labels, predicted_outliers_idx):
    """
    Calculate precision, recall, and F1-score.
    
    Parameters:
    -----------
    true_labels : array-like
        True labels (0=normal, 1=outlier)
    predicted_outliers_idx : list or set
        Indices of predicted outliers
    """
    # Create prediction array
    predicted = np.zeros(len(true_labels))
    predicted[list(predicted_outliers_idx)] = 1
    
    # Calculate metrics
    true_positives = np.sum((predicted == 1) & (true_labels == 1))
    false_positives = np.sum((predicted == 1) & (true_labels == 0))
    false_negatives = np.sum((predicted == 0) & (true_labels == 1))
    
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'true_positives': true_positives,
        'false_positives': false_positives,
        'false_negatives': false_negatives,
        'n_detected': int(true_positives + false_positives)
    }

## Method 1: Z-Score (Univariate Combined) - **SOLUTION**

In [ ]:
# SOLUTION: Apply Z-score to both features and combine
outliers_f1, _ = zscore(challenge_df['Feature_1'], threshold=3)
outliers_f2, _ = zscore(challenge_df['Feature_2'], threshold=3)
method_1_outliers_idx = set(outliers_f1.index) | set(outliers_f2.index)

metrics_1 = calculate_metrics(challenge_df['true_label'], method_1_outliers_idx)
print("Method 1 Results (Z-score Univariate):")
print(f"Detected: {metrics_1['n_detected']} outliers")
print(f"Precision: {metrics_1['precision']:.3f}")
print(f"Recall: {metrics_1['recall']:.3f}")
print(f"F1-Score: {metrics_1['f1_score']:.3f}")
print(f"True Positives: {metrics_1['true_positives']}")
print(f"False Positives: {metrics_1['false_positives']}")
print(f"False Negatives: {metrics_1['false_negatives']}")

## Method 2: Mahalanobis Distance - **SOLUTION**

In [ ]:
# SOLUTION: Apply Mahalanobis Distance
method_2_outliers, _ = detect_mahalanobis_outliers(challenge_df[['Feature_1', 'Feature_2']])
method_2_outliers_idx = set(method_2_outliers.index)

metrics_2 = calculate_metrics(challenge_df['true_label'], method_2_outliers_idx)
print("\nMethod 2 Results (Mahalanobis Distance):")
print(f"Detected: {metrics_2['n_detected']} outliers")
print(f"Precision: {metrics_2['precision']:.3f}")
print(f"Recall: {metrics_2['recall']:.3f}")
print(f"F1-Score: {metrics_2['f1_score']:.3f}")
print(f"True Positives: {metrics_2['true_positives']}")
print(f"False Positives: {metrics_2['false_positives']}")
print(f"False Negatives: {metrics_2['false_negatives']}")

## Method 3: Isolation Forest - **SOLUTION**

In [ ]:
# SOLUTION: Apply Isolation Forest with appropriate contamination
expected_contamination = 50 / 1050  # True outlier rate
method_3_outliers, _, _ = detect_isolation_forest_outliers(
    challenge_df[['Feature_1', 'Feature_2']], 
    contamination=expected_contamination
)
method_3_outliers_idx = set(method_3_outliers.index)

metrics_3 = calculate_metrics(challenge_df['true_label'], method_3_outliers_idx)
print("\nMethod 3 Results (Isolation Forest):")
print(f"Detected: {metrics_3['n_detected']} outliers")
print(f"Precision: {metrics_3['precision']:.3f}")
print(f"Recall: {metrics_3['recall']:.3f}")
print(f"F1-Score: {metrics_3['f1_score']:.3f}")
print(f"True Positives: {metrics_3['true_positives']}")
print(f"False Positives: {metrics_3['false_positives']}")
print(f"False Negatives: {metrics_3['false_negatives']}")

## Comprehensive Comparison - **SOLUTION**

In [ ]:
# SOLUTION: Create comparison visualization and summary table
import pandas as pd

# Summary table
comparison_df = pd.DataFrame([
    {'Method': 'Z-score (Univariate)', **metrics_1},
    {'Method': 'Mahalanobis Distance', **metrics_2},
    {'Method': 'Isolation Forest', **metrics_3}
])

print("\n" + "="*80)
print("COMPREHENSIVE METHOD COMPARISON")
print("="*80)
print(comparison_df.to_string(index=False))
print("\n")

# Visual comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Ground truth
colors_true = ['blue' if label == 0 else 'red' for label in challenge_df['true_label']]
axes[0, 0].scatter(challenge_df['Feature_1'], challenge_df['Feature_2'], 
                   c=colors_true, alpha=0.6, s=30)
axes[0, 0].set_xlabel('Feature 1')
axes[0, 0].set_ylabel('Feature 2')
axes[0, 0].set_title(f'Ground Truth (50 true outliers)', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Method 1: Z-score
normal_1 = challenge_df[~challenge_df.index.isin(method_1_outliers_idx)]
outliers_1 = challenge_df[challenge_df.index.isin(method_1_outliers_idx)]
axes[0, 1].scatter(normal_1['Feature_1'], normal_1['Feature_2'], 
                   c='lightgray', alpha=0.5, s=20, label='Normal')
# Color by correctness
correct_1 = outliers_1[outliers_1['true_label'] == 1]
incorrect_1 = outliers_1[outliers_1['true_label'] == 0]
axes[0, 1].scatter(correct_1['Feature_1'], correct_1['Feature_2'], 
                   c='green', s=50, label='True Positive', edgecolors='black', linewidth=1)
axes[0, 1].scatter(incorrect_1['Feature_1'], incorrect_1['Feature_2'], 
                   c='orange', s=50, label='False Positive', edgecolors='black', linewidth=1)
axes[0, 1].set_xlabel('Feature 1')
axes[0, 1].set_ylabel('Feature 2')
axes[0, 1].set_title(f'Z-score: P={metrics_1["precision"]:.2f}, R={metrics_1["recall"]:.2f}, F1={metrics_1["f1_score"]:.2f}', 
                     fontsize=12, fontweight='bold')
axes[0, 1].legend(loc='upper right')
axes[0, 1].grid(True, alpha=0.3)

# Method 2: Mahalanobis
normal_2 = challenge_df[~challenge_df.index.isin(method_2_outliers_idx)]
outliers_2 = challenge_df[challenge_df.index.isin(method_2_outliers_idx)]
axes[1, 0].scatter(normal_2['Feature_1'], normal_2['Feature_2'], 
                   c='lightgray', alpha=0.5, s=20, label='Normal')
correct_2 = outliers_2[outliers_2['true_label'] == 1]
incorrect_2 = outliers_2[outliers_2['true_label'] == 0]
axes[1, 0].scatter(correct_2['Feature_1'], correct_2['Feature_2'], 
                   c='green', s=50, label='True Positive', edgecolors='black', linewidth=1)
axes[1, 0].scatter(incorrect_2['Feature_1'], incorrect_2['Feature_2'], 
                   c='orange', s=50, label='False Positive', edgecolors='black', linewidth=1)
axes[1, 0].set_xlabel('Feature 1')
axes[1, 0].set_ylabel('Feature 2')
axes[1, 0].set_title(f'Mahalanobis: P={metrics_2["precision"]:.2f}, R={metrics_2["recall"]:.2f}, F1={metrics_2["f1_score"]:.2f}', 
                     fontsize=12, fontweight='bold')
axes[1, 0].legend(loc='upper right')
axes[1, 0].grid(True, alpha=0.3)

# Method 3: Isolation Forest
normal_3 = challenge_df[~challenge_df.index.isin(method_3_outliers_idx)]
outliers_3 = challenge_df[challenge_df.index.isin(method_3_outliers_idx)]
axes[1, 1].scatter(normal_3['Feature_1'], normal_3['Feature_2'], 
                   c='lightgray', alpha=0.5, s=20, label='Normal')
correct_3 = outliers_3[outliers_3['true_label'] == 1]
incorrect_3 = outliers_3[outliers_3['true_label'] == 0]
axes[1, 1].scatter(correct_3['Feature_1'], correct_3['Feature_2'], 
                   c='green', s=50, label='True Positive', edgecolors='black', linewidth=1)
axes[1, 1].scatter(incorrect_3['Feature_1'], incorrect_3['Feature_2'], 
                   c='orange', s=50, label='False Positive', edgecolors='black', linewidth=1)
axes[1, 1].set_xlabel('Feature 1')
axes[1, 1].set_ylabel('Feature 2')
axes[1, 1].set_title(f'Isolation Forest: P={metrics_3["precision"]:.2f}, R={metrics_3["recall"]:.2f}, F1={metrics_3["f1_score"]:.2f}', 
                     fontsize=12, fontweight='bold')
axes[1, 1].legend(loc='upper right')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
methods = ['Z-score', 'Mahalanobis', 'Isolation Forest']
metrics_list = [metrics_1, metrics_2, metrics_3]

# Precision
axes[0].bar(methods, [m['precision'] for m in metrics_list], color=['skyblue', 'coral', 'lightgreen'])
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision Comparison', fontweight='bold')
axes[0].set_ylim([0, 1.1])
for i, v in enumerate([m['precision'] for m in metrics_list]):
    axes[0].text(i, v + 0.02, f"{v:.3f}", ha='center', fontweight='bold')

# Recall
axes[1].bar(methods, [m['recall'] for m in metrics_list], color=['skyblue', 'coral', 'lightgreen'])
axes[1].set_ylabel('Recall')
axes[1].set_title('Recall Comparison', fontweight='bold')
axes[1].set_ylim([0, 1.1])
for i, v in enumerate([m['recall'] for m in metrics_list]):
    axes[1].text(i, v + 0.02, f"{v:.3f}", ha='center', fontweight='bold')

# F1-Score
axes[2].bar(methods, [m['f1_score'] for m in metrics_list], color=['skyblue', 'coral', 'lightgreen'])
axes[2].set_ylabel('F1-Score')
axes[2].set_title('F1-Score Comparison', fontweight='bold')
axes[2].set_ylim([0, 1.1])
for i, v in enumerate([m['f1_score'] for m in metrics_list]):
    axes[2].text(i, v + 0.02, f"{v:.3f}", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## Challenge Analysis - **SOLUTION**

**Which method worked best?**

**SOLUTION:** Based on the results, **Isolation Forest** typically performs best overall, achieving the highest F1-score. However, the "best" method depends on your priorities:

- **Isolation Forest** achieves the best balance (highest F1-score) and excellent recall
- **Mahalanobis Distance** provides good performance with strong statistical interpretation
- **Z-score (Univariate)** has lower recall because it misses multivariate outliers

**Why did it work better than the others?**

**SOLUTION:** Isolation Forest excels because:

1. **No distributional assumptions:** The outliers were uniformly distributed, not normally distributed. Isolation Forest doesn't assume any particular distribution.

2. **Handles complexity:** It can detect outliers based on both univariate extremes AND unusual multivariate combinations.

3. **Non-linear boundaries:** Unlike Mahalanobis (which assumes elliptical contours), Isolation Forest can handle irregular outlier patterns.

4. **Contamination control:** We could set the contamination parameter to match the true outlier rate (~4.8%), giving it an advantage.

However, Mahalanobis performs nearly as well and offers better interpretability!

**What trade-offs did you observe between precision and recall?**

**SOLUTION:**

- **Z-score (Univariate):** Often has decent precision but poor recall. It confidently identifies obvious outliers but misses contextual ones. Conservative approach.

- **Mahalanobis Distance:** Balanced precision and recall. The chi-square threshold (95th percentile) provides a good default, but can be tuned.

- **Isolation Forest:** With proper contamination setting, achieves high recall without sacrificing precision. However, setting contamination too high increases false positives.

**Key insight:** You can always trade precision for recall by adjusting thresholds:
- Lower threshold → more detections → higher recall, lower precision
- Higher threshold → fewer detections → lower recall, higher precision

**In a real-world scenario, would you prioritize precision or recall for outlier detection? Why?**

**SOLUTION:** It depends on the application:

**Prioritize RECALL (catch all outliers) when:**
- **Fraud detection:** Missing one fraud case is very costly
- **Medical diagnosis:** Missing a disease is dangerous
- **Security threats:** Better safe than sorry
- **Quality control for critical systems:** Can't afford defects (aircraft parts)
- **Cost of investigation is low:** Can manually review false positives

**Prioritize PRECISION (minimize false alarms) when:**
- **Customer service:** Don't want to incorrectly flag good customers
- **Automated systems:** False alarms waste resources
- **High investigation cost:** Each alert requires expensive manual review
- **User experience:** Too many false alerts cause alert fatigue
- **Manufacturing:** Scrapping good products is expensive

**Best practice:** Use F1-score as a starting point, but adjust based on business context. Consider the asymmetric costs of errors in your specific application.

**Instructor Note:** Real-world systems often use ensemble methods combining multiple approaches, or use different methods at different stages (high recall for screening, high precision for final decisions).

---

# Summary

Congratulations! You've completed the Foundational Statistical Outlier Detection activity.

## Key Takeaways:

1. **Visualization is essential** for understanding your data and identifying potential outliers
2. **Tukey's IQR method** is robust and doesn't assume normality
3. **Z-score methods** work well for normal distributions but can be sensitive to outliers
4. **Modified Z-score** is more robust than standard Z-score
5. **Multivariate methods** (Mahalanobis, Isolation Forest) are necessary when features are correlated
6. **No single method is perfect** - always compare multiple approaches
7. **Domain knowledge** is crucial for interpreting results and setting thresholds

## Next Steps:

- Explore time-series outlier detection methods
- Learn about ensemble methods combining multiple approaches
- Study deep learning approaches for anomaly detection
- Apply these methods to your own datasets!

---

## Instructor Notes:

**Grading Rubric:**
- Function implementations (40%): Correct implementation of IQR, Z-score, Modified Z-score, Mahalanobis, and Isolation Forest
- Exercise completion (30%): All exercises completed with reasonable answers
- Challenge exercise (20%): Applied multiple methods and provided thoughtful analysis
- Reflection questions (10%): Demonstrated understanding of trade-offs and method selection

**Common Student Mistakes:**
1. Forgetting absolute value in Z-score outlier detection
2. Not understanding that MAD uses median of deviations, not mean
3. Trying to use covariance matrix on Series instead of DataFrame
4. Confusion about contamination parameter in Isolation Forest
5. Mixing up when to use univariate vs multivariate methods

**Extensions:**
- Add Local Outlier Factor (LOF) method
- Explore DBSCAN for density-based outlier detection
- Apply methods to high-dimensional data
- Discuss temporal aspects (outliers that change over time)